# RSNA Knee — cache build `p1` (CPU only)

**Run this on a Kaggle notebook with `Accelerator: None`.** Decoding DICOM is CPU work; running it
on a GPU session would bill the scarce GPU quota (`AGENTS.md` §8) for something a CPU does just as
fast.

**What it does.** Decodes every study into the fixed `[12, 3, 192, 192]` uint8 tensor the model
eats, and saves one `.npz` per study. Save the notebook version, then attach its output as a
dataset to `baseline-v1.ipynb` (`CACHE_INPUT_DIRS`) — training then skips decoding entirely and
every later experiment reuses it.

**The preprocessing cells below are copied verbatim from `baseline-v1.ipynb`.** A hash check at the
end fails loudly if they ever drift: a cache built by different code than the one that reads it is
worse than no cache.

**Chunking.** `CHUNK_INDEX` / `N_CHUNKS` split the work across sessions if one session cannot
finish. Each chunk is its own dataset; attach them all.


In [ ]:
# ================================================================== CONFIG — the only cell to edit
from __future__ import annotations
import glob, json, os, re, time, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODE = "full"          # "smoke" (minutes, proves it runs) | "full" (the baseline) | "submit" (inference only)

# ---- competition constants (rules.md) -----------------------------------------------------------
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_LABEL = len(TARGETS)
SUBMISSION_NAME = "submission.csv"        # rules.md: hard requirement
KAGGLE_LIMIT_H  = 9.0                     # rules.md: CPU or GPU notebook <= 9 h
WORKING_LIMIT_H = 6.75                    # rules.md: 9 h minus 25% headroom

# ---- versioned artefacts (rules.md: cache is keyed by preprocessing version) --------------------
LABELLER_VERSION  = "v1-keyword"          # weak-label rules; bump when the labeller changes
PREPROC_VERSION   = "p1"                  # bump on ANY change to the DICOM -> tensor path
EXPERIMENT_ID     = "baseline-v1"

# ---- study -> tensor geometry -------------------------------------------------------------------
SLOTS       = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1)]   # (plane, prefer fluid-sensitive)
N_SLOT      = len(SLOTS)
N_TRIPLET   = 4            # windows per slot; a window = 3 adjacent slices stacked as channels
IMG         = 192
CROP_MM     = 130.0
SLICE_BAND  = (0.15, 0.85)
K           = N_SLOT * N_TRIPLET

# ---- model / training ---------------------------------------------------------------------------
BACKBONE     = "resnet18"
PRETRAINED   = True        # with internet off this needs an attached weights dataset; see below
EPOCHS       = 4           # FIXED. Never chosen by looking at gold (rules.md hard rule 2)
BATCH        = 8
LR_HEAD      = 3e-4
LR_BACKBONE  = 1e-4
NUM_WORKERS  = 2

# ---- evaluation protocol (rules.md: statistical rules) ------------------------------------------
N_SEEDS   = {"smoke": 1, "full": 3, "submit": 1}[RUN_MODE]   # training seeds -> seed variance
SEEDS     = [2026, 2027, 2028][:N_SEEDS]
N_FOLDS   = 5              # evaluation folds over the 58 gold studies (NOT training folds)
EVAL_REPEATS = 5           # fold reshuffles; sigma comes from (seed x repeat x fold) cells
# Decoding DICOM is the real cost (CPU, ~1-4 s/study cold). 1200 studies keeps the FIRST run inside
# one session; raise it once the cache is built and saved as a Kaggle dataset — see plans/baseline-v1.md.
MAX_TRAIN_STUDIES = {"smoke": 120, "full": 1200, "submit": 0}[RUN_MODE]
CACHE_INPUT_DIRS  = ["/kaggle/input/rsna-knee-cache-p1"]   # prebuilt tensor cache, if attached

# ---- offline weights (rules.md: internet is disabled in the rerun) ------------------------------
# Attach a public timm-weights dataset and point this at it, or accept random init (and say so).
TIMM_OFFLINE_DIRS = ["/kaggle/input/timm-weights", "/kaggle/input/pytorch-image-models-weights"]
CHECKPOINT_DIRS   = ["/kaggle/input/rsna-knee-baseline-v1"]   # our own trained weights, for "submit"

# ---- paths ---------------------------------------------------------------------------------------
def find_root() -> Path:
    for c in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"), Path(".")]:
        if (c / "train.csv").exists() or list(c.glob("train*.csv")):
            return c
    raise FileNotFoundError("Competition data not found; set ROOT by hand.")

def find_csv(root: Path, stem: str) -> Path:
    exact = root / f"{stem}.csv"
    if exact.exists():
        return exact
    hits = sorted(c for c in root.glob(f"{stem}*.csv") if "_series" not in c.name)
    if not hits:
        raise FileNotFoundError(f"{stem}.csv not found under {root}")
    return hits[0]

ROOT  = find_root()
WORK  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / f"cache_{PREPROC_VERSION}"          # version in the path: a stale cache cannot be reused
CACHE.mkdir(parents=True, exist_ok=True)
# A prebuilt cache attached as a read-only dataset is searched first, then the writable one.
CACHE_READ = [Path(d) for d in CACHE_INPUT_DIRS if Path(d).exists()] + [CACHE]

T_START = time.time()
def elapsed_h() -> float:
    return (time.time() - T_START) / 3600.0

def budget_check(stage: str) -> None:
    """rules.md hard rule 7: stay inside the 6.75 h working limit, loudly."""
    h = elapsed_h()
    print(f"[budget] {stage}: {h:.2f} h of {WORKING_LIMIT_H} h used ({h / KAGGLE_LIMIT_H:.0%} of the Kaggle cap)")
    if h > WORKING_LIMIT_H:
        warnings.warn(f"OVER THE WORKING BUDGET at '{stage}' — this configuration is not submittable.")

np.random.seed(SEEDS[0])
print(f"run mode   : {RUN_MODE}   seeds={SEEDS}   epochs={EPOCHS}")
print(f"data root  : {ROOT.resolve()}")
print(f"work dir   : {WORK.resolve()}")
print(f"cache      : {CACHE.name}  (preproc {PREPROC_VERSION}, labeller {LABELLER_VERSION})")
print(f"per study  : {K} windows ({N_SLOT} slots x {N_TRIPLET} triplets) at {IMG}x{IMG}")


In [ ]:
train        = pd.read_csv(find_csv(ROOT, "train"))
train_series = pd.read_csv(find_csv(ROOT, "train_series"))
test_series  = pd.read_csv(find_csv(ROOT, "test_series"))
test         = pd.read_csv(find_csv(ROOT, "test"))
for df in (train, test, train_series, test_series):
    for col in ("StudyInstanceUID", "SeriesInstanceUID"):
        if col in df.columns:
            df[col] = df[col].astype(str)

gold_mask = train[TARGETS].notna().all(axis=1)
gold = train.loc[gold_mask].reset_index(drop=True)
Y_GOLD = gold[TARGETS].values.astype(int)          # [58, 12] — the only ground truth we own

pos = pd.Series(Y_GOLD.sum(0), index=TARGETS)
print(f"gold studies: {len(gold)}   report-only: {(~gold_mask).sum()}")
print("\npositives per target among the gold studies:")
print(pos.to_string())
print(f"\nrarest target: {pos.idxmin()} with {pos.min()} positives -> "
      f"{pos.min() / N_FOLDS:.1f} expected positives per fold. "
      "This is why undefined (fold, label) cells are unavoidable and must be counted.")


In [ ]:
import pydicom
import cv2
cv2.setNumThreads(1)                       # parallelism is across studies, not inside OpenCV

SERIES_DIR_TRAIN = ROOT / "train_series"
SERIES_DIR_TEST  = ROOT / "test_series"
HAVE_IMAGES = SERIES_DIR_TRAIN.exists() or SERIES_DIR_TEST.exists()
print("image directories present:", HAVE_IMAGES)
if not HAVE_IMAGES:
    print("  -> metadata-only environment: training is skipped, the fallback submission stands.")

series_by_study      = {k: v.to_dict("records") for k, v in train_series.groupby("StudyInstanceUID")}
series_by_study_test = {k: v.to_dict("records") for k, v in test_series.groupby("StudyInstanceUID")}

def pick_series(rows, plane, fluid, used):
    cand = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]
    pref = [r for r in cand if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
    return (pref or cand or [None])[0]

def ordered_slices(series_dir: Path):
    keyed = []
    for f in glob.glob(str(series_dir / "*.dcm")):
        try:
            hdr = pydicom.dcmread(f, stop_before_pixels=True)
            pos = int(getattr(hdr, "InstanceNumber", 0) or 0)
            spacing = float(hdr.PixelSpacing[0]) if hasattr(hdr, "PixelSpacing") else 0.0
        except Exception:
            continue
        keyed.append((pos, f, spacing))
    keyed.sort()
    return [(f, s) for _, f, s in keyed]

def read_pixels(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr
        return arr
    except Exception:
        return None

def crop_and_resize(arr, spacing):
    h, w = arr.shape
    if spacing <= 0:
        spacing = CROP_MM / max(h, w)
    side = min(int(round(CROP_MM / spacing)), h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return cv2.resize(arr[y0:y0 + side, x0:x0 + side], (IMG, IMG), interpolation=cv2.INTER_AREA)

def build_study(study_uid: str, rows, series_dir: Path):
    """-> (uint8 [K,3,IMG,IMG], bool [K]). A missing or unreadable slot is zeros with mask False."""
    volume = np.zeros((K, 3, IMG, IMG), np.uint8)
    mask = np.zeros(K, bool)
    used, w = set(), 0
    for plane, fluid in SLOTS:
        record = pick_series(rows, plane, fluid, used)
        if record is None:
            w += N_TRIPLET; continue
        used.add(record["SeriesInstanceUID"])
        files = ordered_slices(series_dir / study_uid / record["SeriesInstanceUID"])
        n = len(files)
        if n == 0:
            w += N_TRIPLET; continue
        lo, hi = int(n * SLICE_BAND[0]), max(int(n * SLICE_BAND[1]) - 1, 0)
        centres = np.linspace(lo, max(hi, lo), N_TRIPLET).round().astype(int)
        med = float(np.median([s for _, s in files if s > 0]) if any(s > 0 for _, s in files) else 0.0)
        for centre in centres:
            idx = [int(np.clip(centre + d, 0, n - 1)) for d in (-1, 0, 1)]
            planes, spacings = [], []
            for i in idx:
                path, spacing = files[i]
                planes.append(read_pixels(path))
                spacings.append(spacing if spacing > 0 else med)
            present = [p for p in planes if p is not None]
            if not present:
                w += 1; continue
            low, high = np.percentile(np.concatenate([p.ravel() for p in present]), [2.0, 98.0])
            for c, (p, spacing) in enumerate(zip(planes, spacings)):
                if p is None:
                    continue
                norm = np.clip((p - low) / (high - low + 1e-6), 0, 1)
                volume[w, c] = (crop_and_resize(norm, spacing) * 255).astype(np.uint8)
            mask[w] = True
            w += 1
    return volume, mask

def cached_study(study_uid: str, rows, series_dir: Path):
    """DICOM decoding, not the GPU, is the bottleneck — build once per (study, PREPROC_VERSION)."""
    for d in CACHE_READ:                          # attached prebuilt cache first, then our own
        path = d / f"{study_uid}.npz"
        if path.exists():
            try:
                with np.load(path) as z:
                    return z["volume"], z["mask"]
            except Exception:
                if d == CACHE:
                    path.unlink(missing_ok=True)  # corrupt entry we own: rebuild rather than crash
    path = CACHE / f"{study_uid}.npz"
    volume, mask = build_study(study_uid, rows, series_dir)
    np.savez_compressed(path, volume=volume, mask=mask)
    return volume, mask


## Build

`N_PROC` workers decode in parallel — this is the whole reason the stage is cheap. Studies already
cached are skipped, so a session that times out can be resumed by simply re-running.


In [ ]:
# ---------------------------------------------------------------- chunking / parallelism
N_CHUNKS    = 1          # raise if one session cannot finish; each chunk -> its own dataset
CHUNK_INDEX = 0
N_PROC      = max(os.cpu_count() or 2, 2)
TIME_LIMIT_H = 8.0       # stop cleanly before the Kaggle session cap and keep what was built

from concurrent.futures import ProcessPoolExecutor, as_completed

all_uids = sorted(set(train["StudyInstanceUID"]) | set(test["StudyInstanceUID"]))
chunk = [u for i, u in enumerate(all_uids) if i % N_CHUNKS == CHUNK_INDEX]
todo  = [u for u in chunk if not (CACHE / f"{u}.npz").exists()]
print(f"studies total {len(all_uids)} | this chunk {len(chunk)} | already cached {len(chunk) - len(todo)} | to build {len(todo)}")
print(f"workers: {N_PROC}")


def _build_one(uid: str):
    """Decode one study into the cache. Returns (uid, valid_windows, seconds, error)."""
    t0 = time.time()
    try:
        if uid in series_by_study:
            rows, sdir = series_by_study[uid], SERIES_DIR_TRAIN
        else:
            rows, sdir = series_by_study_test.get(uid, []), SERIES_DIR_TEST
        volume, mask = build_study(uid, rows, sdir)
        np.savez_compressed(CACHE / f"{uid}.npz", volume=volume, mask=mask)
        return uid, int(mask.sum()), time.time() - t0, ""
    except Exception as e:                      # one bad study must not stop the build
        return uid, 0, time.time() - t0, f"{type(e).__name__}: {e}"


In [ ]:
records, errors, stopped_early = [], [], False
if HAVE_IMAGES and todo:
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=N_PROC) as pool:
        futures = {pool.submit(_build_one, u): u for u in todo}
        for n, fut in enumerate(as_completed(futures), 1):
            uid, valid, secs, err = fut.result()
            records.append({"uid": uid, "valid_windows": valid, "seconds": round(secs, 2)})
            if err:
                errors.append({"uid": uid, "error": err})
            if n % 100 == 0 or n == len(todo):
                done_s = time.time() - t0
                rate = n / max(done_s, 1e-9)
                print(f"  {n}/{len(todo)}  {done_s / 60:.1f} min elapsed  "
                      f"{rate * 60:.0f} studies/min  eta {(len(todo) - n) / max(rate, 1e-9) / 60:.0f} min  "
                      f"errors {len(errors)}")
            if elapsed_h() > TIME_LIMIT_H:
                print("TIME LIMIT — stopping cleanly; re-run this notebook to resume where it stopped.")
                stopped_early = True
                for f in futures:
                    f.cancel()
                break
elif not HAVE_IMAGES:
    print("No image directories here — this notebook only does anything on Kaggle with the data attached.")
else:
    print("Nothing to build: the chunk is already cached.")


## Verify, then save the version

The manifest is what makes the cache trustworthy later: which studies exist, how many valid windows
each has, and — critically — **the hash of the preprocessing code that produced them**.


In [ ]:
import hashlib
import inspect

built = sorted(CACHE.glob("*.npz"))
sizes = np.array([p.stat().st_size for p in built], float)

# Spot-check: a cached tensor must load and have the exact contract shape.
if built:
    with np.load(built[0]) as z:
        v, m = z["volume"], z["mask"]
    assert v.shape == (K, 3, IMG, IMG) and v.dtype == np.uint8, f"bad cached tensor {v.shape} {v.dtype}"
    assert m.shape == (K,), f"bad mask shape {m.shape}"
    print(f"spot check ok: {built[0].name} -> {v.shape} {v.dtype}, {int(m.sum())}/{K} valid windows")

def preproc_fingerprint() -> str:
    """Hash the code that produced these tensors, so a reader can prove the cache matches it.
    `inspect.getsource` works in a notebook kernel; if it cannot (some exec contexts), fall back
    to the geometry constants, which is weaker but never wrong."""
    try:
        src = "".join(inspect.getsource(f) for f in
                      (build_study, crop_and_resize, ordered_slices, pick_series, read_pixels))
        how = "source"
    except (OSError, TypeError):
        src = repr((K, IMG, SLOTS, N_TRIPLET, CROP_MM, SLICE_BAND, PREPROC_VERSION))
        how = "geometry-only"
    return f"{hashlib.sha256(src.encode()).hexdigest()[:16]}:{how}"

PREPROC_SOURCE_HASH = preproc_fingerprint()

valid = pd.DataFrame(records)["valid_windows"] if records else pd.Series(dtype=float)
manifest = {
    "preproc_version": PREPROC_VERSION,
    "preproc_source_sha256_16": PREPROC_SOURCE_HASH,
    "geometry": {"K": K, "IMG": IMG, "slots": SLOTS, "n_triplet": N_TRIPLET,
                 "crop_mm": CROP_MM, "slice_band": list(SLICE_BAND)},
    "chunk_index": CHUNK_INDEX, "n_chunks": N_CHUNKS,
    "studies_cached": len(built),
    "studies_expected_total": len(all_uids),
    "stopped_early": stopped_early,
    "cache_bytes": int(sizes.sum()) if len(sizes) else 0,
    "cache_gb": round(float(sizes.sum()) / 1e9, 2) if len(sizes) else 0.0,
    "mean_seconds_per_study": round(float(np.mean([r["seconds"] for r in records])), 2) if records else None,
    "studies_with_zero_valid_windows": int((valid == 0).sum()) if len(valid) else 0,
    "errors": errors[:50], "n_errors": len(errors),
    "build_hours": round(elapsed_h(), 2),
}
with open(WORK / f"cache_manifest_{PREPROC_VERSION}_chunk{CHUNK_INDEX}.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2)[:2000])

if manifest["cache_gb"] > 18:
    warnings.warn("Cache is near Kaggle's 20 GB output limit — raise N_CHUNKS and split it.")
if manifest["studies_with_zero_valid_windows"]:
    warnings.warn(f"{manifest['studies_with_zero_valid_windows']} studies decoded to ZERO valid "
                  "windows — check the series directory layout before trusting this cache.")
print("\nNext: Save Version -> then attach this notebook's output to baseline-v1.ipynb "
      "via CACHE_INPUT_DIRS, and set PRETRAINED weights dataset while you are there.")
